# Autokeras

AutoKeras is an AutoML (Automated Machine Learning) library built on TensorFlow and Keras, designed to facilitate the creation and optimization of deep learning models. Its main goal is to automate the process of searching for neural network architectures, eliminating the need for manual intervention in choosing hyperparameters and network layers.

python 3.9.20
tensorflow==2.10.1
autokeras==1.1.0
numpy==1.26.4
keras-nlp==0.10.0

In [5]:
import pandas as pd
import numpy as np
import sys
import io
import random
import time

import autokeras as ak
import tensorflow as tf
from sklearn.metrics import classification_report, f1_score, hamming_loss

from pipeline_utils import (
    DEFAULT_K_VALUES,
    compute_multilabel_metrics,
    evaluate_baselines,
    export_experiment_artifacts,
    load_split_csv,
    prepare_multilabel_targets,
)

try:
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
except Exception as e:
    print(f"Could not set UTF-8 encoding: {e}")

def set_seed(seed_value=42):
    np.random.seed(seed_value)
    random.seed(seed_value)
    tf.random.set_seed(seed_value)
    try:
        tf.config.experimental.enable_op_determinism()
    except AttributeError:
        pass

set_seed(42)
print("Random seeds (random, numpy, tensorflow) fixed to 42.")


Could not set UTF-8 encoding: 'OutStream' object has no attribute 'buffer'
Random seeds (random, numpy, tensorflow) fixed to 42.


In [6]:
print("Loading train.csv and test.csv...")
train_df = load_split_csv("train.csv")
test_df = load_split_csv("test.csv")

x_train = np.array(train_df['combinedText'].astype(str).tolist(), dtype=str)
x_test = np.array(test_df['combinedText'].astype(str).tolist(), dtype=str)
y_train, y_test, classes, mlb = prepare_multilabel_targets(train_df, test_df)

print(f"Training samples: {len(x_train)}")
print(f"Test samples: {len(x_test)}")
print(f"Number of classes: {len(classes)}")

baseline_metrics, baseline_predictions, baseline_timing = evaluate_baselines(
    y_train,
    y_test,
    labels=classes,
    k_values=DEFAULT_K_VALUES,
    seed=42,
)
export_experiment_artifacts(
    results_dir="results/autokeras",
    method="baselines",
    metrics_df=baseline_metrics,
    predictions_df=baseline_predictions,
    timing_rows=baseline_timing.to_dict("records"),
    config={
        "notebook": "autokeras.ipynb",
        "seed": 42,
        "k_values": DEFAULT_K_VALUES,
        "labels": list(classes),
        "n_train": int(y_train.shape[0]),
        "n_test": int(y_test.shape[0]),
        "baselines": ["frequency_topk", "random_distribution"],
    },
)
baseline_metrics


Loading train.csv and test.csv...
Training samples: 270
Test samples: 68
Number of classes: 53


,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,baseline_frequency_topk,1,0.181818,0.009224,0.054939,0.0,0.323529,0.129797,0.323529,0.344820,23.529412,68,53
1,baseline_frequency_topk,3,0.238095,0.019934,0.079911,0.0,0.220588,0.282878,0.397059,0.344820,23.529412,68,53
2,baseline_frequency_topk,5,0.229572,0.026978,0.109878,0.0,0.173529,0.392058,0.573529,0.344820,23.529412,68,53
3,baseline_frequency_topk,7,0.215385,0.032597,0.141509,0.0,0.147059,0.443284,0.632353,0.344820,23.529412,68,53
4,baseline_frequency_topk,10,0.189696,0.038300,0.192009,0.0,0.119118,0.535307,0.735294,0.344820,23.529412,68,53
5,baseline_random_distribution,1,0.157025,0.019749,0.056604,0.0,0.279412,0.114601,0.279412,0.262877,28.161765,68,53
6,baseline_random_distribution,3,0.164021,0.022607,0.087680,0.0,0.151961,0.196954,0.367647,0.262877,28.161765,68,53
7,baseline_random_distribution,5,0.175097,0.042265,0.117647,0.0,0.132353,0.259320,0.470588,0.262877,28.161765,68,53
8,baseline_random_distribution,7,0.175385,0.057509,0.148724,0.0,0.119748,0.333836,0.573529,0.262877,28.161765,68,53
9,baseline_random_distribution,10,0.156909,0.058278,0.199778,0.0,0.098529,0.395117,0.602941,0.262877,28.161765,68,53


In [7]:
print("\n--- Training AutoKeras TextClassifier ---")

input_node = ak.TextInput()
output_node = ak.TextBlock(
    block_type="vanilla",
    max_tokens=5000,
)(input_node)

output_node = ak.ClassificationHead(
    multi_label=True,
)(output_node)

clf = ak.AutoModel(
    inputs=input_node,
    outputs=output_node,
    max_trials=10,
    objective="val_loss",
    overwrite=True,
    project_name="autokeras_fair_comparison_v4",
    seed=42,
)

train_start = time.perf_counter()
clf.fit(x_train, y_train, epochs=10, batch_size=32)
train_seconds = time.perf_counter() - train_start
print("AutoKeras training completed.")

infer_start = time.perf_counter()
y_proba = np.asarray(clf.predict(x_test), dtype=float)
inference_seconds = time.perf_counter() - infer_start

thresholds = np.arange(0.05, 1.0, 0.05)
best_thresholds = []
for i in range(y_test.shape[1]):
    y_true_col = y_test[:, i]
    y_proba_col = y_proba[:, i]
    best_f1 = 0
    best_thresh = 0.5
    for thresh in thresholds:
        y_pred_col = (y_proba_col > thresh).astype(int)
        score = f1_score(y_true_col, y_pred_col, average='binary', zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresh = thresh
    best_thresholds.append(float(best_thresh))

y_pred_optimized = np.zeros(y_test.shape)
for i in range(y_test.shape[1]):
    y_pred_optimized[:, i] = (y_proba[:, i] > best_thresholds[i]).astype(int)

print(f"Hamming Loss (optimized thresholds): {hamming_loss(y_test, y_pred_optimized):.4f}")
print(f"F1 Score micro (optimized thresholds): {f1_score(y_test, y_pred_optimized, average='micro'):.4f}")
print(f"F1 Score macro (optimized thresholds): {f1_score(y_test, y_pred_optimized, average='macro', zero_division=0):.4f}")
print(classification_report(y_test, y_pred_optimized, target_names=classes, zero_division=0))

metrics_df, predictions_df = compute_multilabel_metrics(
    y_test,
    y_proba,
    labels=classes,
    method="autokeras",
    k_values=DEFAULT_K_VALUES,
)
paths = export_experiment_artifacts(
    results_dir="results/autokeras",
    method="autokeras",
    metrics_df=metrics_df,
    predictions_df=predictions_df,
    timing_rows=[{
        "method": "autokeras",
        "train_seconds": train_seconds,
        "inference_seconds": inference_seconds,
        "inference_seconds_per_sample": inference_seconds / max(1, len(x_test)),
    }],
    config={
        "notebook": "autokeras.ipynb",
        "seed": 42,
        "k_values": DEFAULT_K_VALUES,
        "labels": list(classes),
        "n_train": int(y_train.shape[0]),
        "n_test": int(y_test.shape[0]),
        "autokeras": {"max_trials": 10, "epochs": 10, "batch_size": 32, "objective": "val_loss"},
        "thresholds": best_thresholds,
    },
)
print(metrics_df)
print("Exported artifacts:", paths)


Trial 10 Complete [00h 00m 02s]
val_loss: 0.2299339473247528

Best val_loss So Far: 0.212539941072464
Total elapsed time: 00h 00m 20s
Epoch 1/10
9/9 [==============================] - 0s 12ms/step - loss: 0.6818 - accuracy: 0.0037
Epoch 2/10
9/9 [==============================] - 0s 14ms/step - loss: 0.6005 - accuracy: 0.0037
Epoch 3/10
9/9 [==============================] - 0s 13ms/step - loss: 0.5186 - accuracy: 0.0037
Epoch 4/10
9/9 [==============================] - 0s 13ms/step - loss: 0.4489 - accuracy: 0.0111
Epoch 5/10
9/9 [==============================] - 0s 13ms/step - loss: 0.3805 - accuracy: 0.0037
Epoch 6/10
9/9 [==============================] - 0s 13ms/step - loss: 0.3165 - accuracy: 0.0037
Epoch 7/10
9/9 [==============================] - 0s 14ms/step - loss: 0.2657 - accuracy: 0.0037
Epoch 8/10
9/9 [==============================] - 0s 12ms/step - loss: 0.2294 - accuracy: 0.0185
Epoch 9/10
9/9 [==============================] - 0s 14ms/step - loss: 0.2071 - accuracy: 

INFO:tensorflow:Assets written to: .\autokeras_fair_comparison_v4\best_model\assets


INFO:tensorflow:Assets written to: .\autokeras_fair_comparison_v4\best_model\assets


AutoKeras training completed.
3/3 [==============================] - 0s 3ms/step
Hamming Loss (optimized thresholds): 0.0549
F1 Score micro (optimized thresholds): 0.1818
F1 Score macro (optimized thresholds): 0.0092
                                                     precision    recall  f1-score   support

                Acessibilidade e inclusão - Docente       0.00      0.00      0.00         7
                              Análise de evidências       0.00      0.00      0.00         2
                               Análise de problemas       0.00      0.00      0.00         1
                         Aprendizagem autorregulada       0.00      0.00      0.00         2
                          Aprendizagem colaborativa       0.00      0.00      0.00         2
                           Colaboração profissional       0.00      0.00      0.00         2
                                        Comunicação       0.00      0.00      0.00         2
                  Comunicação e colabo

In [8]:
# Results are exported in the training cell above.
# Re-run that cell after changing AutoKeras parameters to refresh metrics, sensitivity, timing, predictions, and config files.


In [10]:
experiments = [
    {"name": "autokeras_vanilla", "block_type": "vanilla", "max_trials": 10},
    {"name": "autokeras_transformer", "block_type": "transformer", "max_trials": 10},
    # {"name": "autokeras_bert", "block_type": "bert", "max_trials": 10},
    {"name": "autokeras_ngram", "block_type": "ngram", "max_trials": 10},
]

all_metrics = []

for exp in experiments:
    print(f"\n--- Training {exp['name']} ---")

    input_node = ak.TextInput()
    output_node = ak.TextBlock(
        block_type=exp["block_type"],
        max_tokens=5000,
    )(input_node)

    output_node = ak.ClassificationHead(
        multi_label=True,
    )(output_node)

    clf = ak.AutoModel(
        inputs=input_node,
        outputs=output_node,
        max_trials=exp["max_trials"],
        objective="val_loss",
        overwrite=True,
        project_name=f"{exp['name']}_v1",
        seed=42,
    )

    train_start = time.perf_counter()
    clf.fit(x_train, y_train, epochs=10, batch_size=32)
    train_seconds = time.perf_counter() - train_start

    infer_start = time.perf_counter()
    y_proba = np.asarray(clf.predict(x_test), dtype=float)
    inference_seconds = time.perf_counter() - infer_start

    metrics_df, predictions_df = compute_multilabel_metrics(
        y_test,
        y_proba,
        labels=classes,
        method=exp["name"],
        k_values=DEFAULT_K_VALUES,
    )

    all_metrics.append(metrics_df)

    paths = export_experiment_artifacts(
        results_dir=f"results/{exp['name']}",
        method=exp["name"],
        metrics_df=metrics_df,
        predictions_df=predictions_df,
        timing_rows=[{
            "method": exp["name"],
            "train_seconds": train_seconds,
            "inference_seconds": inference_seconds,
            "inference_seconds_per_sample": inference_seconds / max(1, len(x_test)),
        }],
        config={
            "notebook": "autokeras.ipynb",
            "seed": 42,
            "k_values": DEFAULT_K_VALUES,
            "labels": list(classes),
            "n_train": int(y_train.shape[0]),
            "n_test": int(y_test.shape[0]),
            "autokeras": {
                "block_type": exp["block_type"],
                "max_tokens": 5000,
                "max_trials": exp["max_trials"],
                "epochs": 10,
                "batch_size": 32,
                "objective": "val_loss",
            },
        },
    )

    print(metrics_df)
    print("Exported artifacts:", paths)

comparison_df = pd.concat(all_metrics, ignore_index=True)
comparison_df


Trial 10 Complete [00h 00m 01s]
val_loss: 0.6802101135253906

Best val_loss So Far: 0.2044081836938858
Total elapsed time: 00h 00m 11s
Epoch 1/10
9/9 [==============================] - 0s 3ms/step - loss: 3.1028 - accuracy: 0.1259
Epoch 2/10
9/9 [==============================] - 0s 2ms/step - loss: 6.1315 - accuracy: 0.1926
Epoch 3/10
9/9 [==============================] - 0s 3ms/step - loss: 4.9533 - accuracy: 0.0815
Epoch 4/10
9/9 [==============================] - 0s 3ms/step - loss: 1.4708 - accuracy: 0.0741
Epoch 5/10
9/9 [==============================] - 0s 2ms/step - loss: 0.6505 - accuracy: 0.1963
Epoch 6/10
9/9 [==============================] - 0s 2ms/step - loss: 0.2606 - accuracy: 0.1630
Epoch 7/10
9/9 [==============================] - 0s 3ms/step - loss: 0.2065 - accuracy: 0.1852
Epoch 8/10
9/9 [==============================] - 0s 2ms/step - loss: 0.3645 - accuracy: 0.1963
Epoch 9/10
9/9 [==============================] - 0s 2ms/step - loss: 0.2056 - accuracy: 0.1963
E

INFO:tensorflow:Assets written to: .\autokeras_ngram_v1\best_model\assets


3/3 [==============================] - 0s 1ms/step
            method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0  autokeras_ngram   1  0.181818  0.009224      0.054939              0.0   
1  autokeras_ngram   3  0.132275  0.010819      0.091010              0.0   
2  autokeras_ngram   5  0.120623  0.013878      0.125416              0.0   
3  autokeras_ngram   7  0.110769  0.016522      0.160377              0.0   
4  autokeras_ngram  10  0.110070  0.022295      0.210877              0.0   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.323529     0.129797          0.323529  0.171972            53.0   
1        0.122549     0.149405          0.367647  0.171972            53.0   
2        0.091176     0.204663          0.441176  0.171972            53.0   
3        0.075630     0.230510          0.455882  0.171972            53.0   
4        0.069118     0.289123          0.529412  0.171972            53.0   

   n_samples  n_l

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,autokeras_vanilla,1,0.181818,0.009224,0.054939,0.0,0.323529,0.129797,0.323529,0.171972,53.0,68,53
1,autokeras_vanilla,3,0.132275,0.010819,0.091010,0.0,0.122549,0.149405,0.367647,0.171972,53.0,68,53
2,autokeras_vanilla,5,0.120623,0.013878,0.125416,0.0,0.091176,0.204663,0.441176,0.171972,53.0,68,53
3,autokeras_vanilla,7,0.110769,0.016522,0.160377,0.0,0.075630,0.230510,0.455882,0.171972,53.0,68,53
4,autokeras_vanilla,10,0.110070,0.022295,0.210877,0.0,0.069118,0.289123,0.529412,0.171972,53.0,68,53
5,autokeras_transformer,1,0.181818,0.009224,0.054939,0.0,0.323529,0.129797,0.323529,0.171972,53.0,68,53
6,autokeras_transformer,3,0.132275,0.010819,0.091010,0.0,0.122549,0.149405,0.367647,0.171972,53.0,68,53
7,autokeras_transformer,5,0.120623,0.013878,0.125416,0.0,0.091176,0.204663,0.441176,0.171972,53.0,68,53
8,autokeras_transformer,7,0.110769,0.016522,0.160377,0.0,0.075630,0.230510,0.455882,0.171972,53.0,68,53
9,autokeras_transformer,10,0.110070,0.022295,0.210877,0.0,0.069118,0.289123,0.529412,0.171972,53.0,68,53
